# Multi-GPU Labs — free 2× T4 on Kaggle
### Scale Out · DDP · FSDP · NCCL · Tensor Parallel

**Before you run:** Settings (right panel) → **Accelerator → GPU T4 ×2**. Then **Run All**.

Each lab is written to a `.py` file, then launched across **both T4s** with `torchrun`. You don't edit anything — just run, read the output, and record the numbers for the assignment.

Notes for T4s:
- They use **FP16** (no BF16) — the labs switch automatically.
- Sizes are tuned for **16 GB** cards via command-line flags below.
- The **1-GPU FSDP cell is *meant* to run out of memory** — that's the lesson.

## 0 · Check your two GPUs

In [ ]:
import torch
print("torch:", torch.__version__)
print("GPUs :", torch.cuda.device_count(), "->", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("BF16 supported:", torch.cuda.is_bf16_supported(), "(False on T4 -> labs use FP16)")
assert torch.cuda.device_count() == 2, "Set Accelerator to 'GPU T4 x2' and restart the session."
print("Ready. Two GPUs detected.")

## 1 · Shared model (written to `model.py`)
Both training labs import this.

In [ ]:
%%writefile model.py
# Shared mini-GPT used by both the DDP and FSDP labs.
# Same architecture as the Optimization Sprint, sized up so multi-GPU actually matters.
import math
from dataclasses import dataclass
import torch, torch.nn as nn, torch.nn.functional as F
import torch.utils.checkpoint as cp


@dataclass
class GPTConfig:
    vocab: int = 50304
    seq: int = 512
    dim: int = 2048
    n_layer: int = 16
    n_head: int = 16
    use_ckpt: bool = False        # activation checkpointing (trade compute for memory)


class CausalSelfAttention(nn.Module):
    def __init__(self, c: GPTConfig):
        super().__init__()
        assert c.dim % c.n_head == 0
        self.nh, self.hd = c.n_head, c.dim // c.n_head
        self.qkv = nn.Linear(c.dim, 3 * c.dim)
        self.proj = nn.Linear(c.dim, c.dim)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.nh, self.hd).transpose(1, 2)
        k = k.view(B, T, self.nh, self.hd).transpose(1, 2)
        v = v.view(B, T, self.nh, self.hd).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)   # FlashAttention
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, c: GPTConfig):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(c.dim), nn.LayerNorm(c.dim)
        self.attn = CausalSelfAttention(c)
        self.mlp = nn.Sequential(nn.Linear(c.dim, 4 * c.dim), nn.GELU(), nn.Linear(4 * c.dim, c.dim))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, c: GPTConfig):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab, c.dim)
        self.pos = nn.Embedding(c.seq, c.dim)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.lnf = nn.LayerNorm(c.dim)
        self.head = nn.Linear(c.dim, c.vocab, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))[None]
        for b in self.blocks:
            if self.c.use_ckpt and self.training:
                x = cp.checkpoint(b, x, use_reentrant=False)
            else:
                x = b(x)
        logits = self.head(self.lnf(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.c.vocab), targets.reshape(-1))
        return logits, loss


def count_params(c: GPTConfig) -> int:
    per_layer = 12 * c.dim * c.dim                       # attn(4·dim²) + mlp(8·dim²)
    embeds = 2 * c.vocab * c.dim + c.seq * c.dim         # token + head + positions
    return per_layer * c.n_layer + embeds

## Lab H · DDP — does 2 GPUs give 2×?
Run on **1 GPU**, then **2**, and compute **scaling efficiency = tps(2) / (2 × tps(1))**. Below 1.0 = the gradient all-reduce tax (Kaggle T4s talk over PCIe, no NVLink).

In [ ]:
%%writefile train_ddp.py
# Lab H — DistributedDataParallel on the 2×5090.
# Same model on every GPU, split the data, all-reduce the gradients.
# Run on 1 GPU, then 2, and compute the scaling efficiency (the PCIe tax).
#
#   torchrun --standalone --nproc_per_node=1 train_ddp.py
#   torchrun --standalone --nproc_per_node=2 train_ddp.py
#
import os, time, argparse, torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from model import MiniGPT, GPTConfig, count_params


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dim", type=int, default=2048)
    ap.add_argument("--layers", type=int, default=16)
    ap.add_argument("--heads", type=int, default=16)
    ap.add_argument("--seq", type=int, default=512)
    ap.add_argument("--batch", type=int, default=8)      # per-GPU batch (keep fixed across runs!)
    ap.add_argument("--steps", type=int, default=30)
    ap.add_argument("--warmup", type=int, default=8)
    args = ap.parse_args()

    # ── join the process group; one process per GPU ──
    dist.init_process_group("nccl")
    rank, world = dist.get_rank(), dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
    torch.manual_seed(1234 + rank)                       # different data per rank

    cfg = GPTConfig(dim=args.dim, n_layer=args.layers, n_head=args.heads, seq=args.seq)
    model = MiniGPT(cfg).to(device)
    if rank == 0:
        print(f"[DDP] world_size={world}  params={count_params(cfg)/1e6:.0f}M  per-GPU batch={args.batch}")

    # ── wrap in DDP: this inserts the gradient all-reduce into backward ──
    model = DDP(model, device_ids=[local_rank])
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    bf16 = torch.cuda.get_device_capability(local_rank)[0] >= 8   # BF16 on Ampere+, else FP16
    amp = torch.bfloat16 if bf16 else torch.float16
    scaler = torch.amp.GradScaler(enabled=(amp == torch.float16))

    def get_batch():
        d = torch.randint(0, cfg.vocab, (args.batch, cfg.seq + 1), device=device)
        return d[:, :-1], d[:, 1:]

    def step():
        x, y = get_batch()
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=amp):
            _, loss = model(x, y)
        scaler.scale(loss).backward()      # DDP all-reduces grads here, overlapped with backward
        scaler.step(opt); scaler.update()
        return loss

    # ── warm up, then time the steady state ──
    for _ in range(args.warmup):
        step()
    torch.cuda.synchronize(); dist.barrier()
    torch.cuda.reset_peak_memory_stats(local_rank)
    t0 = time.perf_counter()
    for _ in range(args.steps):
        loss = step()
    torch.cuda.synchronize(); dist.barrier()
    dt = time.perf_counter() - t0

    tokens = world * args.steps * args.batch * args.seq      # total tokens across all GPUs
    peak = torch.cuda.max_memory_allocated(local_rank) / 1e9
    print(f"[rank {rank}] peak VRAM {peak:.1f} GB")
    if rank == 0:
        print(f"[DDP] {world} GPU(s) | {tokens/dt:,.0f} tokens/sec | {dt/args.steps*1e3:.1f} ms/step | loss {loss.item():.2f}")
        print(f"      scaling efficiency = tokens/sec(2 GPU) / (2 × tokens/sec(1 GPU))")
        print(f"      anything < 1.0 is the all-reduce tax — on PCIe (no NVLink) expect ~0.8–0.9")
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
!torchrun --standalone --nproc_per_node=1 train_ddp.py --dim 1024 --layers 12

In [ ]:
!torchrun --standalone --nproc_per_node=2 train_ddp.py --dim 1024 --layers 12

## Lab I · FSDP — fit a model too big for one T4
The model below (~1.4B params) **won't fit one 16 GB T4** but **fits when sharded across two**. The first cell is **expected to OOM** — that's the whole point; the second cell trains it.

In [ ]:
%%writefile train_fsdp.py
# Lab I — FSDP: shard a model that does NOT fit on one 5090.
# The default config (~2.7B params) needs ~44 GB for full training (16 B/param) — it OOMs on
# a single 32 GB card. FSDP shards params + grads + optimizer across both GPUs so it fits.
#
#   # this OOMs (model too big for one card):
#   torchrun --standalone --nproc_per_node=1 train_fsdp.py
#   # this works — sharded across both GPUs:
#   torchrun --standalone --nproc_per_node=2 train_fsdp.py --ckpt
#
# The lecture showed the FSDP2 one-liner `fully_shard(...)`. Here we use the well-established
# FullyShardedDataParallel wrapper — same idea (FULL_SHARD = ZeRO-3), robust across setups.
import os, time, argparse, functools, torch
import torch.distributed as dist
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    ShardingStrategy,
)
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
from model import MiniGPT, GPTConfig, Block, count_params


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dim", type=int, default=2560)      # ~2.7B params → ~44 GB full training
    ap.add_argument("--layers", type=int, default=32)
    ap.add_argument("--heads", type=int, default=20)
    ap.add_argument("--seq", type=int, default=512)
    ap.add_argument("--batch", type=int, default=2)
    ap.add_argument("--steps", type=int, default=20)
    ap.add_argument("--warmup", type=int, default=6)
    ap.add_argument("--ckpt", action="store_true", help="activation checkpointing (use if you OOM)")
    args = ap.parse_args()

    dist.init_process_group("nccl")
    rank, world = dist.get_rank(), dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    torch.cuda.set_device(local_rank)
    torch.manual_seed(1234 + rank)

    cfg = GPTConfig(dim=args.dim, n_layer=args.layers, n_head=args.heads, seq=args.seq, use_ckpt=args.ckpt)
    if rank == 0:
        p = count_params(cfg)
        print(f"[FSDP] world_size={world}  params={p/1e9:.2f}B  full training ≈ {p*16/1e9:.0f} GB (16 B/param)")
        print(f"       sharded across {world} GPUs → roughly {p*16/1e9/world:.0f} GB/GPU before activations")

    model = MiniGPT(cfg)                                   # build on CPU; FSDP shards onto the GPUs
    bf16 = torch.cuda.get_device_capability(local_rank)[0] >= 8
    mp = MixedPrecision(param_dtype=torch.bfloat16, reduce_dtype=torch.float32,
                        buffer_dtype=torch.bfloat16) if bf16 else None
    # wrap EACH transformer block as its own shard unit → gather one block at a time
    policy = functools.partial(transformer_auto_wrap_policy, transformer_layer_cls={Block})

    model = FSDP(
        model,
        auto_wrap_policy=policy,
        mixed_precision=mp,
        sharding_strategy=ShardingStrategy.FULL_SHARD,    # ZeRO-3: shard params, grads AND optimizer
        device_id=local_rank,
        use_orig_params=True,
    )
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    device = torch.device("cuda", local_rank)

    def get_batch():
        d = torch.randint(0, cfg.vocab, (args.batch, cfg.seq + 1), device=device)
        return d[:, :-1], d[:, 1:]

    def step():
        x, y = get_batch()
        opt.zero_grad(set_to_none=True)
        _, loss = model(x, y)              # FSDP casts to bf16 + all-gathers each block on demand
        loss.backward()                    # reduce-scatter grads back to shards
        opt.step()
        return loss

    for _ in range(args.warmup):
        step()
    torch.cuda.synchronize(); dist.barrier()
    torch.cuda.reset_peak_memory_stats(local_rank)
    t0 = time.perf_counter()
    for _ in range(args.steps):
        loss = step()
    torch.cuda.synchronize(); dist.barrier()
    dt = time.perf_counter() - t0

    tokens = world * args.steps * args.batch * args.seq
    peak = torch.cuda.max_memory_allocated(local_rank) / 1e9
    print(f"[rank {rank}] peak VRAM {peak:.1f} GB")
    if rank == 0:
        print(f"[FSDP] {world} GPU(s) | {tokens/dt:,.0f} tokens/sec | {dt/args.steps*1e3:.1f} ms/step | loss {loss.item():.2f}")
        print(f"       compare peak VRAM/GPU here to the ~44 GB the full model would need on one card")
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
# EXPECTED TO OOM on one T4 — this proves the model doesn't fit on a single card.
!torchrun --standalone --nproc_per_node=1 train_fsdp.py --dim 2048 --layers 24 || echo '>>> OOM as expected — now try 2 GPUs below.'

In [ ]:
!torchrun --standalone --nproc_per_node=2 train_fsdp.py --dim 2048 --layers 24 --ckpt

## Lab J · NCCL — measure the real GPU-to-GPU link
An all-reduce micro-benchmark. On Kaggle the two T4s talk over **PCIe** (no NVLink), so the **busbw** will be far below a datacenter GPU's ~900 GB/s.

In [ ]:
%%writefile bench_nccl.py
# Lab J — measure the real GPU-to-GPU link with an all-reduce micro-benchmark.
# Pure PyTorch (no nccl-tests build). Shows the PCIe ceiling on the 2×5090.
#
#   torchrun --standalone --nproc_per_node=2 bench_nccl.py
#
# busbw (bus bandwidth) is the fair number to compare across GPU counts:
#   for ring all-reduce, busbw = algbw × 2·(N−1)/N
# On PCIe 5.0 (no NVLink) expect tens of GB/s; NVLink/NVSwitch would show hundreds.
import os, time, torch
import torch.distributed as dist


def main():
    dist.init_process_group("nccl")
    rank, world = dist.get_rank(), dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    torch.cuda.set_device(local_rank)

    if rank == 0:
        print(f"NCCL all-reduce · world_size={world}")
        print(f"{'size':>8} | {'time':>9} | {'algbw':>10} | {'busbw':>10}")
        print("-" * 46)

    for mb in [1, 4, 16, 64, 256, 512]:
        n = mb * 1024 * 1024 // 4                     # float32 elements
        t = torch.ones(n, device="cuda")
        for _ in range(5):                            # warmup
            dist.all_reduce(t)
        torch.cuda.synchronize(); dist.barrier()

        iters = 20
        t0 = time.perf_counter()
        for _ in range(iters):
            dist.all_reduce(t)
        torch.cuda.synchronize()
        dt = (time.perf_counter() - t0) / iters

        nbytes = n * 4
        algbw = nbytes / dt / 1e9                     # GB/s moved (as seen by the caller)
        busbw = algbw * 2 * (world - 1) / world       # ring all-reduce bus bandwidth
        if rank == 0:
            print(f"{mb:6d}MB | {dt*1e3:7.2f}ms | {algbw:7.1f}GB/s | {busbw:7.1f}GB/s")

    if rank == 0:
        print("\nThe busbw at the large sizes is your interconnect ceiling.")
        print("PCIe 5.0 ×16 ≈ 64 GB/s peak → expect well under that in practice.")
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
!torchrun --standalone --nproc_per_node=2 bench_nccl.py

## Lab K · Tensor Parallelism from scratch
Split one MLP across the two T4s **by hand** (column-parallel + row-parallel) and prove it equals the single-GPU result — the exact Megatron pattern. One all-reduce for the whole MLP.

In [ ]:
%%writefile tp_from_scratch.py
# Lab K — Tensor Parallelism FROM SCRATCH (2 GPUs).
# Split one MLP across 2 GPUs by hand and prove the result matches a single GPU.
# This is the exact column/row-parallel pattern Megatron-LM uses.
#
#   torchrun --standalone --nproc_per_node=2 tp_from_scratch.py
#
import torch
import torch.distributed as dist
import torch.nn.functional as F


def main():
    dist.init_process_group("nccl")
    rank, world = dist.get_rank(), dist.get_world_size()
    assert world == 2, "run with --nproc_per_node=2"
    torch.cuda.set_device(rank)
    dev = torch.device("cuda", rank)

    ## same seed on BOTH ranks → both build the identical "true" full weights
    torch.manual_seed(0)
    D = 4096
    W1 = torch.randn(D, 4 * D, device=dev) / D ** 0.5          # first layer  (D → 4D)
    W2 = torch.randn(4 * D, D, device=dev) / (4 * D) ** 0.5    # second layer (4D → D)
    x = torch.randn(8, D, device=dev)                         # same input on both ranks

    ## ── reference: the whole MLP on ONE GPU (both ranks compute it identically) ──
    ref = F.gelu(x @ W1) @ W2

    ## ── the tensor-parallel version, split across the 2 GPUs ──
    half = (4 * D) // world                                    # split the hidden dim in two

    ## 1) COLUMN-parallel first layer: each rank owns HALF the columns of W1.
    ##    No communication — each rank just makes its half of the hidden activations.
    W1_shard = W1[:, rank * half:(rank + 1) * half]           # (D, 2D)
    h_shard = F.gelu(x @ W1_shard)                            # (8, 2D)  — my half of the hidden

    ## 2) ROW-parallel second layer: split W2's ROWS to match the hidden split.
    ##    Each rank makes a PARTIAL output; all-reduce sums them into the full output.
    W2_shard = W2[rank * half:(rank + 1) * half, :]           # (2D, D)
    y_partial = h_shard @ W2_shard                           # (8, D)  — partial sum
    dist.all_reduce(y_partial)                               # ← the ONE collective for the whole MLP
    tp = y_partial

    ## ── verify the split version equals the single-GPU version ──
    err = (tp - ref).abs().max().item()
    scale = ref.abs().mean().item()
    if rank == 0:
        print(f"each GPU stored HALF of W1 ({tuple(W1_shard.shape)}) and HALF of W2 ({tuple(W2_shard.shape)})")
        print(f"max abs error = {err:.2e}   (relative {err/scale:.1e})")
        print("MATCH ✅ — same result, half the weights per GPU." if err / scale < 1e-3
              else "MISMATCH ❌ — check the split.")
        print("Note: the whole MLP needed exactly ONE all-reduce. That's the Megatron trick.")
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
!torchrun --standalone --nproc_per_node=2 tp_from_scratch.py

## What to record for the assignment (Part 1)
| Lab | Number to report | The question |
|---|---|---|
| **DDP** | tokens/sec at 1 & 2 GPUs → **scaling efficiency** | why is it below 1.0? |
| **FSDP** | 1-GPU **OOM** confirmed · peak VRAM/GPU on 2 | what three things got sharded? |
| **NCCL** | **busbw** at 256–512 MB | how far below NVLink's ~900 GB/s, and why? |
| **TP** | max error vs single-GPU (~1e-3) | why does column-parallel need no comm, but row-parallel needs an all-reduce? |

Write **2–3 sentences** explaining each number. Then go do Parts 2–5 (design · investigate · explain · the **3D Parallelism Arena**).